### Importing Libraries

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('ggplot')

### 2. Load Datasets

In [ ]:
trades = pd.read_csv("historical_data.csv")
sentiment = pd.read_csv("fear_greed_index.csv")

#### Data Quality Section

In [ ]:
print("Trader Dataset Shape:", trades.shape)
print("Sentiment Dataset Shape:", sentiment.shape)

print("\nMissing Values:")
print(trades.isnull().sum())

print("\nDuplicate Rows:")
print(trades.duplicated().sum())

Before beginning the analysis, both datasets were reviewed for missing values, duplicate records, and formatting inconsistencies.

The data was generally clean and required minimal preprocessing. Date and timestamp fields were standardized to ensure consistency, after which the trading data was merged with the daily Fear & Greed Index using trade dates. No significant data quality concerns were identified during this process.

### 3. Data Inspection

In [ ]:
print(trades.shape)
print(sentiment.shape)

trades.head()

In [ ]:
trades.info()


In [ ]:
sentiment.head()

In [ ]:
sentiment.info()

### 4. Data Cleaning

#### Trader Dataset

In [ ]:
trades.isnull().sum()

In [ ]:
trades['Timestamp IST'] = pd.to_datetime(
    trades['Timestamp IST'],
    dayfirst=True
)

trades['trade_date'] = trades['Timestamp IST'].dt.date

In [ ]:
sentiment['date'] = pd.to_datetime(sentiment['date'])

sentiment['trade_date'] = sentiment['date'].dt.date

### 5. Merge Datasets


In [ ]:
merged = pd.merge(
    trades,
    sentiment[['trade_date','classification']],
    on='trade_date',
    how='left'
)

In [ ]:
merged.head()

### Analysis 1: Profitability by Market Sentiment

In [ ]:
sentiment_stats = merged.groupby('classification').agg(
    total_trades=('Closed PnL','count'),
    avg_pnl=('Closed PnL','mean'),
    total_pnl=('Closed PnL','sum')
).reset_index()

sentiment_stats

In [ ]:
plt.figure(figsize=(8,5))

sns.barplot(
    data=sentiment_stats,
    x='classification',
    y='avg_pnl'
)

plt.title("Average PnL by Market Sentiment")
plt.show()

FINDING:

The results suggest that market sentiment is associated with noticeable differences in trading profitability.

Among all sentiment categories, Extreme Greed recorded the highest average profit per trade at $67.89, contributing approximately $2.72 million in realized profits across 39,992 trades. This may indicate that traders were able to capitalize more effectively on strong bullish market conditions.

Fear periods generated the highest total realized profit, approximately $3.36 million from 61,837 trades. One possible explanation is that traders were more active during these periods, resulting in a larger overall profit contribution despite a lower average profit per trade.

In contrast, Extreme Fear and Neutral conditions produced the lowest average profits, both around $34 per trade.

Overall, the findings suggest that both fearful and optimistic market environments can create profitable opportunities, although the underlying trader behavior appears to differ. Fear periods were characterized by higher participation and larger aggregate profits, whereas Extreme Greed was associated with stronger profitability on a per-trade basis.

### Analysis 2: Win Rate by Sentiment

Create win flag

In [ ]:
merged['win_trade'] = np.where(
    merged['Closed PnL'] > 0,
    1,
    0
)

Calculate win rates

In [ ]:
win_rate = merged.groupby(
    'classification'
)['win_trade'].mean()*100

win_rate = win_rate.reset_index()

win_rate

Plot

In [ ]:
plt.figure(figsize=(8,5))

sns.barplot(
    data=win_rate,
    x='classification',
    y='win_trade'
)

plt.title("Win Rate by Sentiment")
plt.ylabel("Win Rate %")
plt.show()

FINDING:

The analysis suggests that trade success rates varied across different market sentiment conditions.

Extreme Greed recorded the highest win rate at 46.49%, meaning nearly one out of every two trades closed profitably. This may indicate that traders were generally more successful during periods of strong market optimism.

Fear conditions achieved a win rate of 42.08%, while Extreme Fear recorded the lowest win rate at 37.06%, suggesting that traders faced greater challenges in maintaining consistent profitability during highly pessimistic market environments.

Interestingly, despite having the lowest win rate, Extreme Fear still generated positive overall profits. One possible explanation is that a smaller number of highly profitable trades offset a larger number of losing positions during these periods.

Overall, the findings suggest that market sentiment may influence not only the magnitude of trading profits but also the likelihood of executing successful trades.

### Analysis 3: Position Size Behavior

In [ ]:
position_size = merged.groupby(
    'classification'
)['Size USD'].mean().reset_index()

position_size

#### Plot

In [ ]:
plt.figure(figsize=(8,5))

sns.barplot(
    data=position_size,
    x='classification',
    y='Size USD'
)

plt.title("Average Position Size by Sentiment")
plt.show()

FINDING:

The analysis suggests that trader position sizing varied across different market sentiment conditions.

The largest average position size was observed during Fear periods at approximately $7,816 per trade, followed by Greed at $5,737 and Extreme Fear at $5,350. This may indicate that traders were willing to allocate more capital during periods of uncertainty, possibly in anticipation of stronger price movements or market reversals.

Interestingly, Extreme Greed recorded the smallest average position size at approximately $3,112 per trade, despite producing both the highest average profit per trade and the highest win rate.

One possible interpretation is that favorable market conditions during Extreme Greed allowed traders to achieve stronger results without increasing their average exposure. In contrast, Fear periods were associated with larger capital commitments but comparatively lower profitability on a per-trade basis.

Overall, the findings suggest that market sentiment may influence not only trading outcomes but also how traders manage capital and position sizing decisions.

### Analysis 4: Trader Profitability Distribution

Top performing 10 traders

In [ ]:
top_traders = merged.groupby(
    'Account'
)['Closed PnL'].sum().sort_values(
    ascending=False
).head(10)

top_traders

Plot

In [ ]:
top_traders.plot(
    kind='bar',
    figsize=(10,5)
)

plt.title("Top 10 Traders by Total PnL")
plt.show()

Distribution of Trader Profitability 

In [ ]:
trader_summary = merged.groupby('Account').agg(
    Total_PnL=('Closed PnL','sum'),
    Number_of_Trades=('Closed PnL','count')
)

trader_summary.describe()

In [ ]:
profitable_traders = (
    trader_summary['Total_PnL'] > 0
).mean() * 100

print(
    f"Percentage of profitable traders: {profitable_traders:.2f}%"
)

In [ ]:
plt.figure(figsize=(8,5))

sns.histplot(
    trader_summary['Total_PnL'],
    bins=20,
    kde=True
)

plt.title("Distribution of Trader Profitability")
plt.xlabel("Total PnL")
plt.show()

The distribution of trader profitability is positively skewed, with a small number of traders generating exceptionally high profits while most traders achieved more moderate results.

FINDING:

The analysis suggests that trader profitability was unevenly distributed across the accounts included in the dataset.

The most profitable trader generated approximately $2.14 million in realized profits, while the tenth-ranked trader generated approximately $379,000. This highlights a substantial gap between the highest-performing traders and the rest of the population.

Across all analyzed traders, the average profit was approximately $321,780, whereas the median profit was only $117,655. The difference between the mean and median suggests that overall profitability was influenced by a relatively small number of exceptionally successful traders.

Interestingly, 90.62% of traders were profitable overall, indicating that positive returns were common during the observed period. However, profit distribution remained highly skewed, with a small group of traders accounting for a disproportionately large share of total gains.

Trading activity also varied considerably. The most active trader executed 40,184 trades, compared to a median of 3,699 trades across all traders. This variation may reflect differences in trading style, strategy frequency, or risk appetite.

Overall, the findings suggest that while most traders achieved positive returns, the scale of profitability varied significantly. Consistent performance, trade execution, and risk management may have played an important role in distinguishing top-performing traders from the broader group.

### Analysis 5: Asset-Level Profitability Analysis

In [ ]:
coin_profit = merged.groupby('Coin').agg(
    total_pnl=('Closed PnL','sum')
).sort_values(
    by='total_pnl',
    ascending=False
).head(10)

coin_profit

In [ ]:
plt.figure(figsize=(10,5))

coin_profit['total_pnl'].plot(kind='bar')

plt.title('Top 10 Coins by Total Realized Profit')
plt.ylabel('Total PnL')
plt.show()

In [ ]:
top10_share = (
    coin_profit['total_pnl'].sum() /
    merged['Closed PnL'].sum()
) * 100

print(f"Top 10 coins contribute {top10_share:.2f}% of total profits")

FINDING:

The analysis suggests that profitability was concentrated among a relatively small number of assets.

The token @107 generated the highest total realized profit at approximately $2.78 million, followed by HYPE ($1.95 million), SOL ($1.64 million), ETH ($1.32 million), and BTC ($0.87 million). These assets contributed a substantial share of the overall profits observed in the dataset.

Notably, the top 10 assets accounted for 94.19% of all realized profits. This concentration suggests that a large portion of trading gains came from a limited set of high-performing assets rather than being distributed evenly across the market.

One possible interpretation is that successful traders were selective in the assets they traded, focusing their activity on opportunities that offered stronger risk-reward characteristics or more favorable market conditions.

Overall, the findings suggest that asset selection may have played an important role in trading performance. While market sentiment appears to influence trader behavior and profitability, the choice of asset also appears to be a significant factor in determining overall returns.

----------------------

# Key Takeaways

1. Extreme Greed recorded the highest average profit per trade ($67.89) and the highest win rate (46.49%) among all sentiment categories.

2. Fear periods generated the highest total realized profit ($3.36M) and the largest average position size ($7,816), indicating higher trading activity and capital deployment during uncertain market conditions.

3. Profitability was unevenly distributed across traders. The most profitable trader generated approximately $2.14M in realized profits, while the median trader profit was approximately $117K.

4. Asset-level performance was highly concentrated, with the top 10 assets accounting for 94.19% of all realized profits generated in the dataset.

5. The findings suggest that market sentiment is associated with changes in profitability, win rates, and trader capital allocation behavior.

------------------

# EXECUTIVE SUMMARY

## Objective

The objective of this analysis was to investigate the relationship between Bitcoin market sentiment (Fear & Greed Index) and trader performance using Hyperliquid historical trading data.

A total of 211,224 executed trades across 32 unique trader accounts were analyzed and matched with daily Fear & Greed sentiment classifications. The study focused on understanding how market sentiment is associated with profitability, win rates, position sizing, trader behavior, and asset-level performance.
---

## Key Findings

### 1. Market Sentiment Significantly Influences Trading Performance

Trading profitability varied across different sentiment regimes. Extreme Greed periods produced the highest average profit per trade at $67.89, while Fear periods generated the highest overall realized profit of approximately $3.36 million across 61,837 trades.

This indicates that market sentiment has a measurable impact on trader outcomes and should be considered when developing trading strategies.

---

### 2. Trade Success Rates Improve During Extreme Greed

Extreme Greed recorded the highest win rate at 46.49%, outperforming all other sentiment categories.

In contrast, Extreme Fear produced the lowest win rate at 37.06%, suggesting that traders face greater difficulty achieving consistent success during highly pessimistic market conditions.

However, despite the lower win rate, Extreme Fear periods still generated positive profits overall, indicating the presence of occasional high-return trades.

---

### 3. Traders Commit More Capital During Fear Periods

Position sizing behavior changed noticeably across market conditions.

Fear periods exhibited the largest average position size at approximately $7,816 per trade, while Extreme Greed showed the smallest average position size at approximately $3,112.

This suggests that traders tend to deploy more capital during fearful market conditions, potentially viewing market downturns as buying opportunities.

---

### 4. Trader Profitability Distribution

The most profitable trader generated approximately $2.14 million in realized profits, while the tenth-ranked trader generated approximately $379,000.

The average trader profit was approximately $321,780, whereas the median profit was only $117,655. This gap suggests that overall profitability was influenced by a relatively small number of exceptionally successful traders.

Notably, 90.62% of traders were profitable overall, indicating that positive returns were common during the observed period. However, profit distribution remained uneven, with a small number of traders accounting for a disproportionately large share of total gains.

These findings suggest that while profitability was achievable for most traders, the magnitude of returns varied considerably across participants.
---

### 5. Asset Selection Is a Major Driver of Profitability

Asset-level analysis revealed a strong concentration of profits among a limited number of coins.

The most profitable asset (@107) generated approximately $2.78 million in realized profits, followed by HYPE ($1.95 million), SOL ($1.64 million), ETH ($1.32 million), and BTC ($0.87 million).

Most notably, the top 10 assets accounted for 94.19% of all realized profits in the dataset, demonstrating that successful traders concentrated their activity in a small subset of high-performing assets.

---

## Conclusion

The analysis identified meaningful relationships between market sentiment and trading performance.

Extreme Greed periods were associated with the highest average profitability and win rates, while Fear periods generated the highest trading activity, largest average position sizes, and greatest aggregate profits. The results also revealed substantial concentration of profits among a small number of traders and assets.

Overall, the findings suggest that trader outcomes are influenced by a combination of market conditions, capital allocation decisions, and asset selection. Incorporating sentiment-based signals alongside disciplined risk management may help improve trading performance and decision-making.

---

## Recommendations

1. Incorporate market sentiment indicators into trading decision frameworks.

2. Adjust position sizing dynamically based on prevailing market sentiment and volatility conditions.

3. Prioritize rigorous asset selection, as a small number of assets generated the vast majority of profits.

4. Study the behavior and strategies of consistently profitable traders to identify characteristics associated with superior performance.

5. Combine sentiment analysis, asset selection, and disciplined risk management to improve trading performance and long-term profitability.